In [1]:
import warnings
warnings.simplefilter("ignore")

In [2]:
# %pip install datasets

In [3]:
from datasets import load_dataset

musiccaps = load_dataset("google/MusicCaps")

print(musiccaps)
print(musiccaps["train"][0])

DatasetDict({
    train: Dataset({
        features: ['ytid', 'start_s', 'end_s', 'audioset_positive_labels', 'aspect_list', 'caption', 'author_id', 'is_balanced_subset', 'is_audioset_eval'],
        num_rows: 5521
    })
})
{'ytid': '-0Gj8-vB1q4', 'start_s': 30, 'end_s': 40, 'audioset_positive_labels': '/m/0140xf,/m/02cjck,/m/04rlf', 'aspect_list': "['low quality', 'sustained strings melody', 'soft female vocal', 'mellow piano melody', 'sad', 'soulful', 'ballad']", 'caption': 'The low quality recording features a ballad song that contains sustained strings, mellow piano melody and soft female vocal singing over it. It sounds sad and soulful, like something you would hear at Sunday services.', 'author_id': 4, 'is_balanced_subset': False, 'is_audioset_eval': True}


In [4]:
from pathlib import Path

musiccaps_dir = Path("../dataset/musiccaps")
musiccaps_dir.mkdir(parents=True, exist_ok=True)

musiccaps["train"].to_csv(musiccaps_dir / "musiccaps_metadata.csv", index=False)

print("Saved:", musiccaps_dir / "musiccaps_metadata.csv")
print("Rows:", len(musiccaps["train"]))

Creating CSV from Arrow format: 100%|██████████| 6/6 [00:00<00:00, 96.71ba/s]

Saved: ..\dataset\musiccaps\musiccaps_metadata.csv
Rows: 5521


In [5]:
musiccaps["train"].to_pandas()[["ytid", "start_s", "end_s", "caption"]].head(10)

,ytid,start_s,end_s,caption
0,-0Gj8-vB1q4,30,40,The low quality recording features a ballad so...
1,-0SdAVK79lg,30,40,This song features an electric guitar as the m...
2,-0vPFx-wRRI,30,40,a male voice is singing a melody with changing...
3,-0xzrMun0Rs,30,40,This song contains digital drums playing a sim...
4,-1LrH01Ei1w,30,40,This song features a rubber instrument being p...
5,-1OlgJWehn8,30,40,This clip is three tracks playing consecutivel...
6,-1UWSisR2zo,30,40,A male singer sings this groovy melody. The so...
7,-3Kv4fdm7Uk,30,40,someone is playing a high pitched melody on a ...
8,-4NLarMj4xU,30,40,The Pop song features a soft female vocal sing...
9,-4SYC2YgzL8,30,40,low fidelity audio from a live performance fea...


In [6]:
f'Clip duration: {musiccaps["train"][0]["end_s"] - musiccaps["train"][0]["start_s"]} seconds'

'Clip duration: 10 seconds'

In [7]:
# %pip install yt-dlp

In [8]:
import yt_dlp

print("yt-dlp version:", yt_dlp.version.__version__)

yt-dlp version: 2026.08.19


In [9]:
# from pathlib import Path
# import yt_dlp

# test_dir = Path("../dataset/musiccaps/audio")
# test_dir.mkdir(parents=True, exist_ok=True)

# row = musiccaps["train"][0]
# ytid = row["ytid"]
# start_s = row["start_s"]
# end_s = row["end_s"]

# url = f"https://www.youtube.com/watch?v={ytid}"

# ydl_opts = {
#     "format": "bestaudio/best",
#     "outtmpl": str(test_dir / f"{ytid}.%(ext)s"),
#     "quiet": False,
# }

# with yt_dlp.YoutubeDL(ydl_opts) as ydl:
#     ydl.download([url])

# print("Test download finished.")

In [10]:
# %pip install imageio-ffmpeg

In [11]:
import imageio_ffmpeg

ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()
print("FFmpeg:", ffmpeg_path)

FFmpeg: C:\Users\21101255\AppData\Roaming\Python\Python312\site-packages\imageio_ffmpeg\binaries\ffmpeg-win-x86_64-v7.1.exe


In [12]:
import subprocess
from pathlib import Path
import imageio_ffmpeg

ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()

input_file = Path("../dataset/musiccaps/audio/-0Gj8-vB1q4.webm")
output_file = Path("../dataset/musiccaps/audio/-0Gj8-vB1q4.wav")

row = musiccaps["train"][0]

subprocess.run([
    ffmpeg_path,
    "-y",
    "-ss", str(row["start_s"]),
    "-i", str(input_file),
    "-t", str(row["end_s"] - row["start_s"]),
    "-ar", "22050",
    "-ac", "1",
    str(output_file)
], check=True)

print("Created:", output_file)
print("Duration:", row["end_s"] - row["start_s"], "seconds")

Created: ..\dataset\musiccaps\audio\-0Gj8-vB1q4.wav
Duration: 10 seconds


In [13]:
# from pathlib import Path
# import yt_dlp
# import pandas as pd

# audio_dir = Path("../dataset/musiccaps/audio")
# audio_dir.mkdir(parents=True, exist_ok=True)

# musiccaps_df = musiccaps["train"].to_pandas()

# ydl_opts = {
#     "format": "bestaudio/best",
#     "outtmpl": str(audio_dir / "%(id)s.%(ext)s"),
#     "quiet": True,
#     "no_warnings": True,
# }

# success = 0
# failed = []

# with yt_dlp.YoutubeDL(ydl_opts) as ydl:
#     for i, row in musiccaps_df.iterrows():
#         ytid = row["ytid"]
        
#         existing = list(audio_dir.glob(f"{ytid}.*"))
#         if existing:
#             success += 1
#             continue
        
#         try:
#             ydl.download([f"https://www.youtube.com/watch?v={ytid}"])
#             success += 1
#         except Exception as e:
#             failed.append((ytid, str(e)))

#         if (i + 1) % 100 == 0:
#             print(f"Processed {i + 1}/{len(musiccaps_df)} | Success: {success} | Failed: {len(failed)}")

# print("\nFinished.")
# print("Success:", success)
# print("Failed:", len(failed))

In [14]:
from pathlib import Path

audio_dir = Path("../dataset/musiccaps/audio")

audio_files = list(audio_dir.glob("*"))
webm_files = list(audio_dir.glob("*.webm"))
wav_files = list(audio_dir.glob("*.wav"))

print("Total audio files:", len(audio_files))
print("WEBM files:", len(webm_files))
print("WAV files:", len(wav_files))

Total audio files: 1902
WEBM files: 719
WAV files: 951


In [15]:
# from pathlib import Path
# import subprocess
# import imageio_ffmpeg

# audio_dir = Path("../dataset/musiccaps/audio")
# ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()

# musiccaps_df = musiccaps["train"].to_pandas()

# success = 0
# failed = []

# for _, row in musiccaps_df.iterrows():
#     ytid = row["ytid"]
#     input_files = list(audio_dir.glob(f"{ytid}.*"))
    
#     if not input_files:
#         continue

#     input_file = input_files[0]
#     output_file = audio_dir / f"{ytid}.wav"

#     if output_file.exists():
#         success += 1
#         continue

#     try:
#         subprocess.run([
#             ffmpeg_path,
#             "-y",
#             "-ss", str(row["start_s"]),
#             "-i", str(input_file),
#             "-t", str(row["end_s"] - row["start_s"]),
#             "-ar", "22050",
#             "-ac", "1",
#             str(output_file)
#         ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

#         success += 1

#     except Exception as e:
#         failed.append(ytid)

# print("WAV clips ready:", success)
# print("Failed:", len(failed))

In [16]:
from pathlib import Path
import pandas as pd

audio_dir = Path("../dataset/musiccaps/audio")

musiccaps_df = musiccaps["train"].to_pandas().copy()
musiccaps_df["audio_path"] = musiccaps_df["ytid"].apply(lambda x: str(audio_dir / f"{x}.wav"))
musiccaps_df = musiccaps_df[musiccaps_df["audio_path"].apply(lambda x: Path(x).exists())].reset_index(drop=True)
musiccaps_df = musiccaps_df.sample(frac=1, random_state=42).reset_index(drop=True)

n = len(musiccaps_df)
n_train = int(0.8 * n)
n_val = int(0.1 * n)

musiccaps_train = musiccaps_df.iloc[:n_train].reset_index(drop=True)
musiccaps_val = musiccaps_df.iloc[n_train:n_train+n_val].reset_index(drop=True)
musiccaps_test = musiccaps_df.iloc[n_train+n_val:].reset_index(drop=True)


print("Total paired clips:", len(musiccaps_df))
print("Train:", len(musiccaps_train))
print("Validation:", len(musiccaps_val))
print("Test:", len(musiccaps_test))

Total paired clips: 951
Train: 760
Validation: 95
Test: 96


In [17]:
import torch
import torchaudio
import torch.nn.functional as F
from torch_geometric.data import Data
from pathlib import Path
import numpy as np

print("PyTorch:", torch.__version__)
print("PyG Data:", Data)
print("Train clips:", len(musiccaps_train))

PyTorch: 2.13.0+cu130
PyG Data: <class 'torch_geometric.data.data.Data'>
Train clips: 760


In [18]:
import librosa
import numpy as np
import torch
from torch_geometric.data import Data
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

def build_musiccaps_graph(audio_path):
    y, sr = librosa.load(audio_path, sr=22050, mono=True)

    segment_len = int(3.0 * sr)
    hop_len = int(1.5 * sr)

    node_features = []

    for start in range(0, max(1, len(y) - segment_len + 1), hop_len):
        segment = y[start:start + segment_len]

        if len(segment) < segment_len:
            segment = np.pad(segment, (0, segment_len - len(segment)))

        mfcc_full = librosa.feature.mfcc(y=segment, sr=sr, n_mfcc=20)

        mfcc_mean = mfcc_full.mean(axis=1)
        mfcc_std = mfcc_full.std(axis=1)

        delta_full = librosa.feature.delta(mfcc_full)
        delta_mean = delta_full.mean(axis=1)
        delta_std = delta_full.std(axis=1)

        chroma_full = librosa.feature.chroma_stft(y=segment, sr=sr)
        chroma_mean = chroma_full.mean(axis=1)
        chroma_std = chroma_full.std(axis=1)

        tonnetz_full = librosa.feature.tonnetz(y=segment, sr=sr)
        tonnetz_mean = tonnetz_full.mean(axis=1)
        tonnetz_std = tonnetz_full.std(axis=1)

        contrast_full = librosa.feature.spectral_contrast(y=segment, sr=sr)
        contrast_mean = contrast_full.mean(axis=1)
        contrast_std = contrast_full.std(axis=1)

        centroid = librosa.feature.spectral_centroid(y=segment, sr=sr)
        rolloff = librosa.feature.spectral_rolloff(y=segment, sr=sr)
        zcr = librosa.feature.zero_crossing_rate(segment)
        rms = librosa.feature.rms(y=segment)

        feat = np.concatenate([
            mfcc_mean, mfcc_std,
            delta_mean, delta_std,
            chroma_mean, chroma_std,
            tonnetz_mean, tonnetz_std,
            contrast_mean, contrast_std,
            [centroid.mean(), centroid.std()],
            [rolloff.mean(), rolloff.std()],
            [zcr.mean(), zcr.std()],
            [rms.mean(), rms.std()]
        ])

        node_features.append(feat)

    x = np.asarray(node_features, dtype=np.float32)

    edges = []
    edge_weights = []

    num_nodes = len(x)

    for i in range(num_nodes - 1):
        edges.append([i, i + 1])
        edges.append([i + 1, i])
        edge_weights.extend([1.0, 1.0])

    if num_nodes > 1:
        sim = cosine_similarity(x)
        k = min(3, num_nodes - 1)

        for i in range(num_nodes):
            neighbors = np.argsort(sim[i])[-(k + 1):-1]

            for j in neighbors:
                if i != j:
                    edges.append([i, int(j)])
                    edges.append([int(j), i])
                    edge_weights.extend([float(sim[i, j]), float(sim[i, j])])

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_weights, dtype=torch.float32).view(-1, 1)

    return Data(x=torch.tensor(x, dtype=torch.float32), edge_index=edge_index, edge_attr=edge_attr)


def build_split_graphs(split_df):
    graphs = []
    valid_rows = []

    for _, row in tqdm(split_df.iterrows(), total=len(split_df)):
        try:
            graph = build_musiccaps_graph(row["audio_path"])
            graphs.append(graph)
            valid_rows.append(row)

        except Exception as e:
            print("Failed:", row["ytid"], e)

    valid_df = pd.DataFrame(valid_rows).reset_index(drop=True)

    return graphs, valid_df


musiccaps_train_graphs, musiccaps_train_valid = build_split_graphs(musiccaps_train)
musiccaps_val_graphs, musiccaps_val_valid = build_split_graphs(musiccaps_val)
musiccaps_test_graphs, musiccaps_test_valid = build_split_graphs(musiccaps_test)


print("Train graphs:", len(musiccaps_train_graphs))
print("Validation graphs:", len(musiccaps_val_graphs))
print("Test graphs:", len(musiccaps_test_graphs))
print("Node feature dimension:", musiccaps_train_graphs[0].x.shape[1])

100%|██████████| 96/96 [00:25<00:00,  3.71it/s]

Train graphs: 760
Validation graphs: 95
Test graphs: 96
Node feature dimension: 138


In [19]:
import torch
from pathlib import Path

graph_dir = Path("../dataset/musiccaps")
graph_dir.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        "train_graphs": musiccaps_train_graphs,
        "train_df": musiccaps_train_valid,
        "val_graphs": musiccaps_val_graphs,
        "val_df": musiccaps_val_valid,
        "test_graphs": musiccaps_test_graphs,
        "test_df": musiccaps_test_valid,
    },
    graph_dir / "musiccaps_graphs.pt"
)


print("Saved:", graph_dir / "musiccaps_graphs.pt")

Saved: ..\dataset\musiccaps\musiccaps_graphs.pt


In [25]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

device

device(type='cuda')

In [26]:
from transformers import BertTokenizer, BertModel

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
musiccaps_tokenizer = tokenizer

bert = BertModel.from_pretrained("bert-base-uncased").to(device)
bert.eval()

def get_caption_embeddings(df):
    texts = df["caption"].astype(str).tolist()
    encodings = musiccaps_tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
    embeddings = []

    with torch.no_grad():
        for start in range(0, len(texts), 32):
            batch = {k: v[start:start+32].to(device) for k, v in encodings.items()}
            outputs = bert(**batch)
            embeddings.append(outputs.last_hidden_state.cpu())

    return torch.cat(embeddings, dim=0), encodings


musiccaps_train_bert, train_encodings = get_caption_embeddings(musiccaps_train_valid)
musiccaps_val_bert, val_encodings = get_caption_embeddings(musiccaps_val_valid)
musiccaps_test_bert, test_encodings = get_caption_embeddings(musiccaps_test_valid)


print("Train:", musiccaps_train_bert.shape)
print("Validation:", musiccaps_val_bert.shape)
print("Test:", musiccaps_test_bert.shape)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5827.66it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Train: torch.Size([760, 124, 768])
Validation: torch.Size([95, 124, 768])
Test: torch.Size([96, 115, 768])


In [27]:
import torch.nn as nn
from torch_geometric.nn import GATv2Conv
from torch_geometric.nn import global_mean_pool, global_max_pool


class Task4GNN(nn.Module):
    def __init__(self, in_channels=138, hidden_channels=48, heads=2, dropout=0.5):

        super().__init__()

        self.conv1 = GATv2Conv(
            in_channels,
            hidden_channels,
            heads=heads,
            edge_dim=1,
            dropout=dropout
        )

        self.bn1 = nn.BatchNorm1d(hidden_channels * heads)

        self.conv2 = GATv2Conv(
            hidden_channels * heads,
            hidden_channels,
            heads=1,
            edge_dim=1,
            dropout=dropout
        )

        self.bn2 = nn.BatchNorm1d(hidden_channels)

        self.dropout = dropout

    def forward(self, x, edge_index, batch, edge_attr=None):

        if edge_attr is not None and edge_attr.dim() == 1:
            edge_attr = edge_attr.unsqueeze(1)

        x = self.conv1(
            x,
            edge_index,
            edge_attr=edge_attr
        )

        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        x = self.conv2(
            x,
            edge_index,
            edge_attr=edge_attr
        )

        x = self.bn2(x)
        x = F.relu(x)

        g = torch.cat([
            global_mean_pool(x, batch),
            global_max_pool(x, batch)
        ], dim=1)

        return g


task4_gnn = Task4GNN().to(device)


print("Task 4 GNN created.")
print("Graph embedding dimension:", 48 * 2)

Task 4 GNN created.
Graph embedding dimension: 96


In [29]:
class ProjectionHead(nn.Module):
    def __init__(self, input_dim, output_dim=256):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(512, output_dim)
        )

    def forward(self, x):
        return self.net(x)


gnn_projection = ProjectionHead(96).to(device)
bert_projection = ProjectionHead(768).to(device)


print("GNN projection: 96 -> 256")
print("BERT projection: 768 -> 256")

GNN projection: 96 -> 256
BERT projection: 768 -> 256


In [30]:
import torch.nn.functional as F

def symmetric_info_nce(g_emb, t_emb, temperature=0.07):
    g_emb = F.normalize(g_emb, p=2, dim=1)
    t_emb = F.normalize(t_emb, p=2, dim=1)

    logits = torch.matmul(g_emb, t_emb.T) / temperature
    labels = torch.arange(logits.size(0), device=logits.device)

    loss_g2t = F.cross_entropy(logits, labels)
    loss_t2g = F.cross_entropy(logits.T, labels)

    final_loss = (loss_g2t + loss_t2g) / 2

    return final_loss


print("Symmetric InfoNCE loss ready.")

Symmetric InfoNCE loss ready.


In [31]:
from torch_geometric.data import Batch
from torch.utils.data import DataLoader


train_loader = DataLoader(range(len(musiccaps_train_graphs)), batch_size=16, shuffle=True)
val_loader = DataLoader(range(len(musiccaps_val_graphs)), batch_size=16, shuffle=False)


print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Train batches: 48
Validation batches: 6


In [ ]:
# from pathlib import Path
# import subprocess
# import imageio_ffmpeg

# audio_dir = Path("../dataset/musiccaps/audio")
# ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()

# musiccaps_df = musiccaps["train"].to_pandas()

# success = 0
# failed = []

# for _, row in musiccaps_df.iterrows():
#     ytid = row["ytid"]
#     input_files = list(audio_dir.glob(f"{ytid}.*"))
    
#     if not input_files:
#         continue

#     input_file = input_files[0]
#     output_file = audio_dir / f"{ytid}.wav"

#     if output_file.exists():
#         success += 1
#         continue

#     try:
#         subprocess.run([
#             ffmpeg_path,
#             "-y",
#             "-ss", str(row["start_s"]),
#             "-i", str(input_file),
#             "-t", str(row["end_s"] - row["start_s"]),
#             "-ar", "22050",
#             "-ac", "1",
#             str(output_file)
#         ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

#         success += 1

#     except Exception as e:
#         failed.append(ytid)

# print("WAV clips ready:", success)
# print("Failed:", len(failed))

Fresh Task 4 models created.
Node features standardized using training statistics.
Train batch size: 32
Train batches: 24
Validation batches: 3


In [32]:
from pathlib import Path

audio_dir = Path("../dataset/musiccaps/audio")

audio_files = list(audio_dir.glob("*"))
webm_files = list(audio_dir.glob("*.webm"))
wav_files = list(audio_dir.glob("*.wav"))


print("Total audio files:", len(audio_files))
print("WEBM files:", len(webm_files))
print("WAV files:", len(wav_files))

Total audio files: 1902
WEBM files: 719
WAV files: 951


In [33]:
import torch
import torch.nn as nn

task4_gnn = Task4GNN(in_channels=138, hidden_channels=48, heads=2, dropout=0.3).to(device)

gnn_projection = ProjectionHead(96, 256).to(device)
bert_projection = ProjectionHead(768, 256).to(device)

checkpoint = torch.load("saved_models/t1_bert_best_model.pt", map_location=device, weights_only=True)
bert_state = {k.replace("bert.", "", 1): v for k, v in checkpoint.items() if k.startswith("bert.")}

bert.load_state_dict(bert_state, strict=True)
bert.to(device)

for param in bert.parameters():
    param.requires_grad = False

for layer in bert.encoder.layer[-2:]:
    for param in layer.parameters():
        param.requires_grad = True

optimizer = torch.optim.AdamW(
    [
        {
            "params": task4_gnn.parameters(),
            "lr": 5e-5
        },
        {
            "params": gnn_projection.parameters(),
            "lr": 5e-5
        },
        {
            "params": bert_projection.parameters(),
            "lr": 5e-5
        },
        {
            "params": [
                p for p in bert.parameters()
                if p.requires_grad
            ],
            "lr": 1e-6
        }
    ],
    weight_decay=1e-4
)


print("Fresh Task 4 model created.")
print("BERT trainable layers: last 2")
print("BERT LR: 1e-6")
print("GNN/projection LR: 5e-5")

Fresh Task 4 model created.
BERT trainable layers: last 2
BERT LR: 1e-6
GNN/projection LR: 5e-5


In [34]:
def run_epoch(loader, graphs, valid_indices, training=True):
    if training:
        task4_gnn.train()
        gnn_projection.train()
        bert_projection.train()
        bert.train()

    else:
        task4_gnn.eval()
        gnn_projection.eval()
        bert_projection.eval()
        bert.eval()

    total_loss = 0.0

    for batch_indices in loader:
        batch_indices = batch_indices.tolist()

        graph_batch = Batch.from_data_list([graphs[i].cpu() for i in batch_indices]).to(device)

        captions = [str(valid_indices.iloc[i]["caption"]) for i in batch_indices]

        tokens = tokenizer(captions, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)

        if training:
            optimizer.zero_grad()

        g = task4_gnn(
            graph_batch.x,
            graph_batch.edge_index,
            graph_batch.batch,
            graph_batch.edge_attr
        )

        t = bert(input_ids=tokens["input_ids"], attention_mask=tokens["attention_mask"]).last_hidden_state[:, 0, :]

        g = gnn_projection(g)
        t = bert_projection(t)

        loss = symmetric_info_nce(g, t, temperature=0.07)

        if training:
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                list(task4_gnn.parameters()) +
                list(gnn_projection.parameters()) +
                list(bert_projection.parameters()) +
                [p for p in bert.parameters() if p.requires_grad],
                max_norm=1.0
            )

            optimizer.step()

        total_loss += loss.item()
        final_loss = total_loss / len(loader)

    return final_loss

In [38]:
num_epochs = 30
patience = 5

best_val_loss = float("inf")
epochs_without_improvement = 0

train_losses = []
val_losses = []

for epoch in range(1, num_epochs + 1):

    train_loss = run_epoch(train_loader, musiccaps_train_graphs, musiccaps_train_valid, training=True)
    val_loss = run_epoch(val_loader, musiccaps_val_graphs, musiccaps_val_valid, training=False)
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch:02d}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0

        torch.save({
            "gnn": task4_gnn.state_dict(),
            "gnn_projection": gnn_projection.state_dict(),
            "bert_projection": bert_projection.state_dict(),
            "bert": bert.state_dict(),
        }, "saved_models/t4_best_contrastive_regularized.pt")

        print("  Best model saved")

    else:
        epochs_without_improvement += 1
        print(f"  No improvement: {epochs_without_improvement}/{patience}")

        if epochs_without_improvement >= patience:
            print("  Early stopping triggered.")
            break

print(f"Best validation loss: {best_val_loss:.4f}")

Epoch 01/30 | Train Loss: 2.8096 | Val Loss: 2.7454
  Best model saved
Epoch 02/30 | Train Loss: 2.6487 | Val Loss: 2.7052
  Best model saved
Epoch 03/30 | Train Loss: 2.6037 | Val Loss: 2.6673
  Best model saved
Epoch 04/30 | Train Loss: 2.5643 | Val Loss: 2.6519
  Best model saved
Epoch 05/30 | Train Loss: 2.5183 | Val Loss: 2.6579
  No improvement: 1/5
Epoch 06/30 | Train Loss: 2.5237 | Val Loss: 2.6534
  No improvement: 2/5
Epoch 07/30 | Train Loss: 2.4495 | Val Loss: 2.6472
  Best model saved
Epoch 08/30 | Train Loss: 2.3991 | Val Loss: 2.6507
  No improvement: 1/5
Epoch 09/30 | Train Loss: 2.4018 | Val Loss: 2.6395
  Best model saved
Epoch 10/30 | Train Loss: 2.3600 | Val Loss: 2.6051
  Best model saved
Epoch 11/30 | Train Loss: 2.3524 | Val Loss: 2.6898
  No improvement: 1/5
Epoch 12/30 | Train Loss: 2.3713 | Val Loss: 2.6368
  No improvement: 2/5
Epoch 13/30 | Train Loss: 2.3170 | Val Loss: 2.6573
  No improvement: 3/5
Epoch 14/30 | Train Loss: 2.2711 | Val Loss: 2.6497
  No im

In [50]:
import torch
import torch.nn.functional as F
import numpy as np

checkpoint = torch.load("saved_models/t4_best_contrastive_regularized.pt", map_location=device, weights_only=True)
task4_gnn.load_state_dict(checkpoint["gnn"])
gnn_projection.load_state_dict(checkpoint["gnn_projection"])
bert_projection.load_state_dict(checkpoint["bert_projection"])
bert.load_state_dict(checkpoint["bert"])

task4_gnn.eval()
gnn_projection.eval()
bert_projection.eval()
bert.eval()

test_loader_eval = DataLoader(range(len(musiccaps_test_graphs)), batch_size=32, shuffle=False)

all_g_embs = []
all_t_embs = []

with torch.no_grad():
    for indices in test_loader_eval:
        indices = indices.tolist()
        graph_batch = Batch.from_data_list([musiccaps_test_graphs[i] for i in indices]).to(device)
        captions = [str(musiccaps_test_valid.iloc[i]["caption"]) for i in indices]
        tokens = tokenizer(captions, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)

        g = task4_gnn(graph_batch.x, graph_batch.edge_index, graph_batch.batch, graph_batch.edge_attr)
        t = bert(input_ids=tokens["input_ids"], attention_mask=tokens["attention_mask"]).last_hidden_state[:, 0, :]

        g = F.normalize(gnn_projection(g), p=2, dim=1)
        t = F.normalize(bert_projection(t), p=2, dim=1)

        all_g_embs.append(g.cpu())
        all_t_embs.append(t.cpu())

all_g_embs = torch.cat(all_g_embs, dim=0)
all_t_embs = torch.cat(all_t_embs, dim=0)

sim_matrix = torch.matmul(all_g_embs, all_t_embs.T)

def compute_recall(sim_matrix):
    n = sim_matrix.size(0)
    preds = sim_matrix.argsort(dim=1, descending=True)
    targets = torch.arange(n).view(-1, 1)
    r1 = (preds[:, :1] == targets).any(dim=1).float().mean().item()
    r5 = (preds[:, :5] == targets).any(dim=1).float().mean().item()
    r10 = (preds[:, :10] == targets).any(dim=1).float().mean().item()
    return r1, r5, r10

r1, r5, r10 = compute_recall(sim_matrix)

print("=== MusicCaps Contrastive Retrieval Results ===")
print(f"R@1  : {r1 * 100:.2f}%")
print(f"R@5  : {r5 * 100:.2f}%")
print(f"R@10 : {r10 * 100:.2f}%")

=== MusicCaps Contrastive Retrieval Results ===
R@1  : 3.12%
R@5  : 13.54%
R@10 : 16.67%


In [40]:
import torch
import torch.nn.functional as F

checkpoint = torch.load("saved_models/t4_best_contrastive_regularized.pt", map_location=device, weights_only=True)

task4_gnn.load_state_dict(checkpoint["gnn"])
gnn_projection.load_state_dict(checkpoint["gnn_projection"])
bert_projection.load_state_dict(checkpoint["bert_projection"])
bert.load_state_dict(checkpoint["bert"])

task4_gnn.eval()
gnn_projection.eval()
bert_projection.eval()
bert.eval()

similarity = torch.matmul(all_t_embs, all_g_embs.T)

print("=== 10 Qualitative Retrieval Examples ===\n")

for i in range(min(10, len(musiccaps_test_valid))):
    caption = str(musiccaps_test_valid.iloc[i]["caption"])

    top3 = similarity[i].topk(3).indices.tolist()

    print(f"Example {i+1}")
    print(f"Caption: {caption}")
    print("Top-3 retrieved clips:")

    for rank, idx in enumerate(top3, 1):
        retrieved_caption = str(musiccaps_test_valid.iloc[idx]["caption"])

        print(f"  {rank}. Test index {idx}")
        print(f"     Caption: {retrieved_caption}")


    print("-" * 100)

=== 10 Qualitative Retrieval Examples ===

Example 1
Caption: A group of Arabic folk musicians play this poignant folk music . The song is medium tempo with percussion instruments playing  a steady rhythm, a stringed instrument like violin plays a solo accompanied by other string instruments playing rhythm. The song is poignant and melancholic. The song is a traditional Arabic folk dance song.
Top-3 retrieved clips:
  1. Test index 19
     Caption: The low quality recording features a rock song that consists of flat male vocal singing over sustained and arpeggiated electric guitar melodies, followed by snappy rimshots, soft kick, shimmering hi hats and groovy bass guitar. It sounds emotional and easygoing.
  2. Test index 76
     Caption: This is a rock music piece with a male vocal. The synth and the piano are playing the main melody in a minor scale while the bass guitar is supporting them in the background. The rhythm is played by acoustic drums although it is rather slow. The song 

In [ ]:
import os
os.makedirs("results/task4", exist_ok=True)

qualitative_results = []

for i in range(min(10, len(musiccaps_test_valid))):
    caption = str(musiccaps_test_valid.iloc[i]["caption"])
    top3 = similarity[i].topk(3).indices.tolist()

    qualitative_results.append({
        "Example": i + 1,
        "Query Caption": caption,
        "Retrieved 1": str(musiccaps_test_valid.iloc[top3[0]]["caption"]),
        "Retrieved 2": str(musiccaps_test_valid.iloc[top3[1]]["caption"]),
        "Retrieved 3": str(musiccaps_test_valid.iloc[top3[2]]["caption"])
    })

qualitative_df = pd.DataFrame(qualitative_results)
qualitative_df.to_csv("../results/task4/qualitative_retrieval_examples.csv", index=False)
print("Saved: ../results/task4/qualitative_retrieval_examples.csv")

Saved: ../results/task4/qualitative_retrieval_examples.csv


In [41]:
import pandas as pd

retrieval_results = pd.DataFrame({
    "Metric": ["R@1", "R@5", "R@10"],
    "Score": [
        f"{r1 * 100:.2f}%",
        f"{r5 * 100:.2f}%",
        f"{r10 * 100:.2f}%"
    ]
})


retrieval_results

,Metric,Score
0,R@1,3.12%
1,R@5,13.54%
2,R@10,16.67%


In [48]:
retrieval_results.to_csv("../results/task4/retrieval_metrics.csv", index=False)
print("Saved: ../results/task4/retrieval_metrics.csv")

Saved: ../results/task4/retrieval_metrics.csv


In [42]:
from transformers import BertForSequenceClassification

task3_model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=50, problem_type="multi_label_classification").to(device)
checkpoint = torch.load("saved_models/t1_bert_best_model.pt", map_location=device, weights_only=True)
task3_model.load_state_dict(checkpoint, strict=True)
task3_model.eval()

print()
print("Task 3 supervised model loaded successfully.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6567.63it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi


Task 3 supervised model loaded successfully.


In [43]:
import pandas as pd

annotations = pd.read_csv("../dataset/magnatagatune/annotations_final.csv", sep="\t")
tag_columns = [c for c in annotations.columns if c != "clip_id" and pd.api.types.is_numeric_dtype(annotations[c])]
task3_tags = (annotations[tag_columns].sum().sort_values(ascending=False).head(50).index.tolist())


print("Task 3 labels:", len(task3_tags))

Task 3 labels: 50


In [44]:
import torch
import torch.nn.functional as F
import pandas as pd

checkpoint = torch.load("saved_models/t4_best_contrastive_regularized.pt", map_location=device, weights_only=True)

task4_gnn.load_state_dict(checkpoint["gnn"])
gnn_projection.load_state_dict(checkpoint["gnn_projection"])
bert_projection.load_state_dict(checkpoint["bert_projection"])
bert.load_state_dict(checkpoint["bert"])

task4_gnn.eval()
gnn_projection.eval()
bert_projection.eval()
bert.eval()

common_tags = [
    "rock", "pop", "hip-hop", "electronic",
    "folk", "jazz", "classical", "metal",
    "happy", "sad", "calm", "dark",
    "energetic", "romantic"
]

with torch.no_grad():
    tag_tokens = tokenizer(common_tags, padding=True, truncation=True, max_length=32, return_tensors="pt").to(device)
    tag_cls = bert(input_ids=tag_tokens["input_ids"], attention_mask=tag_tokens["attention_mask"]).last_hidden_state[:, 0, :]
    tag_embeddings = F.normalize(bert_projection(tag_cls), p=2, dim=1)


task4_audio_embeddings = []

with torch.no_grad():
    for graph in musiccaps_test_graphs:
        graph = graph.to(device)
        batch = torch.zeros(graph.x.size(0), dtype=torch.long, device=device)
        g = task4_gnn(graph.x, graph.edge_index, batch, graph.edge_attr)
        g = F.normalize(gnn_projection(g), p=2, dim=1)
        task4_audio_embeddings.append(g.cpu())


task4_audio_embeddings = torch.cat(task4_audio_embeddings).to(device)

task4_similarity = task4_audio_embeddings @ tag_embeddings.T

task4_predictions = []

for i in range(task4_similarity.size(0)):
    top_indices = task4_similarity[i].topk(3).indices
    predicted = [common_tags[j] for j in top_indices.cpu().tolist()]
    task4_predictions.append(predicted)

task3_model.eval()


task3_predictions = []

with torch.no_grad():
    for i in range(len(musiccaps_test_valid)):
        caption = str(musiccaps_test_valid.iloc[i]["caption"])
        tokens = tokenizer(caption, return_tensors="pt", truncation=True, max_length=128).to(device)
        output = task3_model(**tokens)
        probs = torch.sigmoid(output.logits)[0]
        top_indices = probs.topk(3).indices
        predicted = [task3_tags[j] for j in top_indices.cpu().tolist()]
        task3_predictions.append(predicted)


comparison = pd.DataFrame({
    "Caption": [str(musiccaps_test_valid.iloc[i]["caption"]) for i in range(min(10, len(musiccaps_test_valid)))],
    "Task 4 Zero-Shot": task4_predictions[:10],
    "Task 3 Supervised": task3_predictions[:10]
})


comparison

,Caption,Task 4 Zero-Shot,Task 3 Supervised
0,A group of Arabic folk musicians play this poi...,"[folk, rock, pop]","[indian, sitar, drums]"
1,The low quality recording features a cover of ...,"[hip-hop, dark, pop]","[guitar, rock, drums]"
2,The low quality recording features a flat male...,"[pop, hip-hop, folk]","[guitar, drums, rock]"
3,This audio contains a female voice speaking in...,"[folk, metal, rock]","[female, drums, woman]"
4,This is a hip-hop music piece. There is a male...,"[pop, energetic, folk]","[drums, fast, techno]"
5,The low quality video features a filtered male...,"[folk, sad, happy]","[drums, female, vocal]"
6,This song contains a digital drum with a soft ...,"[pop, folk, rock]","[drums, fast, beat]"
7,The low quality recording features a resonatin...,"[calm, metal, jazz]","[drums, guitar, slow]"
8,This is an instrumental power metal piece. The...,"[rock, folk, pop]","[drums, fast, guitar]"
9,This gothic rock song features a male voice si...,"[rock, pop, folk]","[drums, indian, fast]"


In [49]:
comparison.to_csv("../results/task4/task4_vs_task3_comparison.csv", index=False)
print("Saved: ../results/task4/task4_vs_task3_comparison.csv")

Saved: ../results/task4/task4_vs_task3_comparison.csv
